## Exemplo de obtenção de informação dos repositórios

In [ ]:
import requests
import json
import time
import os
from datetime import datetime, timedelta


GITHUB_TOKEN = "os.environ.get("GITHUB_TOKEN", "")"


session = requests.Session()
headers = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
else:
    print("WARNING: No GITHUB_TOKEN found.")
session.headers.update(headers)


def check_rate_limit(response):
    """If we're close to the rate limit, sleep until it resets."""
    remaining = response.headers.get("X-RateLimit-Remaining")
    reset_ts = response.headers.get("X-RateLimit-Reset")
    if remaining is not None and int(remaining) <= 1 and reset_ts:
        wait = int(reset_ts) - int(time.time()) + 2
        if wait > 0:
            print(f"Rate limit nearly exhausted. Sleeping {wait}s until reset...")
            time.sleep(wait)


def log_error(context, response):
    print(f"Error {context}: HTTP {response.status_code} - {response.text[:200]}")


# ---------------------------------------------------------------------------
# SEARCH CONFIG
# ---------------------------------------------------------------------------
search_links = [
    "https://api.github.com/search/repositories?q=serverless+language:python&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=serverless+language:javascript&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=serverless+language:typescript&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=function+as+a+service+language:python&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=function+as+a+service+language:javascript&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=function+as+a+service+language:typescript&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=faas+language:python&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=faas+language:javascript&sort=stars&order=desc&page={}",
    "https://api.github.com/search/repositories?q=faas+language:typescript&sort=stars&order=desc&page={}",
]

one_year_ago = datetime.now() - timedelta(days=365)

# Final array requested: one entry per valid repo, with only the fields needed.
valid_repos = []

for search in search_links:
    print(f"Searching in: {search.split('?q=')[1].split('&')[0]}")

    for page in range(1, 2):
        api_result = session.get(search.format(page))
        check_rate_limit(api_result)

        if api_result.status_code != 200:
            log_error("during search", api_result)
            break

        repos = json.loads(api_result.text)

        for repo in repos.get("items", []):
            repo_owner = repo["owner"]["login"]
            repo_name = repo["name"]
            full_name = repo["full_name"]

            # --- Must be public -----------------------------------------------------
            # The search API only indexes public repos by default, but we check
            # explicitly since you'll be reviewing the code later and a private
            # repo slipping through (e.g. via a token with broader access) would
            # mean you can't actually open it.
            if repo.get("private", False):
                print(f"Skipping {full_name}: repository is private")
                continue

            # --- Last commit date -----------------------------------------------------
            commits_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/commits"
            commits_response = session.get(commits_url, params={"per_page": 1})
            check_rate_limit(commits_response)

            if commits_response.status_code != 200:
                log_error(f"fetching commits for {full_name}", commits_response)
                continue

            commits_data = json.loads(commits_response.text)
            if not commits_data:
                print(f"Skipping {full_name}: no commits returned (empty repo?)")
                continue

            last_commit_date = datetime.strptime(
                commits_data[0]["commit"]["author"]["date"], "%Y-%m-%dT%H:%M:%SZ"
            )

            if last_commit_date < one_year_ago:
                print(f"Skipping {full_name}: last commit older than 1 year "
                      f"({last_commit_date.strftime('%Y-%m-%d')})")
                continue

            # --- Collaborator/contributor count ---------------------------------------
            # /collaborators requires push access to the repo, so it 403s for
            # almost every repo you don't own. /contributors is the public
            # equivalent. If it can't be fetched for any reason, we record
            # None rather than skipping the repo, since you asked for this
            # info "when available".
            contributors_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contributors"
            contributors_response = session.get(
                contributors_url, params={"per_page": 1, "anon": "true"}
            )
            check_rate_limit(contributors_response)

            num_collaborators = None
            if contributors_response.status_code != 200:
                log_error(f"fetching contributors for {full_name}", contributors_response)
            else:
                link_header = contributors_response.headers.get("Link", "")
                if 'rel="last"' in link_header:
                    last_url = [
                        link.split(";")[0].strip("<> ")
                        for link in link_header.split(",")
                        if 'rel="last"' in link
                    ][0]
                    num_collaborators = int(last_url.split("page=")[-1].split("&")[0])
                else:
                    num_collaborators = len(json.loads(contributors_response.text))

            # --- Framework/plugin README filter (kept from prior version) -------------
            included = True
            time.sleep(0.5)
            readme_url = (
                repo["html_url"].replace(
                    "https://github.com", "https://raw.githubusercontent.com"
                )
                + f"/{repo['default_branch']}/README.md"
            )
            readme_response = session.get(readme_url)

            if readme_response.status_code == 200:
                readme_content = readme_response.text.lower()
                if "framework" in readme_content or "plugin" in readme_content:
                    included = False

            if not included:
                continue

            valid_repos.append({
                "url": repo["html_url"],
                "last_commit_date": last_commit_date.strftime("%Y-%m-%d"),
                "num_collaborators": num_collaborators,
            })

            print(
                f"Found: {repo['html_url']} - "
                f"Last commit: {last_commit_date.strftime('%Y-%m-%d')} - "
                f"Collaborators: {num_collaborators}"
            )

        time.sleep(2)

print(f"\nSearch completed! {len(valid_repos)} valid repositories found.")
print(json.dumps(valid_repos, indent=2))


## Exportação dos repositórios para CSV

In [ ]:
import pandas as pd

df_repos = pd.DataFrame(valid_repos)

# Export to CSV, if needed
df_repos.to_csv("valid_repos.csv", sep=";", index=False)
print(df_repos.to_csv(sep=";", index=False))
df_repos

## Extração de métricas dos repositórios

In [ ]:
import os
import stat
import re
import shutil
from git import Repo
import pprint

pp = pprint.PrettyPrinter(indent=4)

def is_config_file(file):
    config_file_rules = [".json", "config.", ".yml", ".conf", ".config", "settings."]
    return any(rule in file for rule in config_file_rules)

def get_func_names(file, language):
    try:
        with open(file, encoding="utf-8") as handle:
            content = handle.read()
    except UnicodeDecodeError as ex:
        return []
        
    if language == "Python":
        regex = "def (.*)\("
    elif language == "JavaScript":
        regex = "exports\.(.*) = .*"

    matches = re.findall(regex, content)

    return matches

def get_func_params(file, language):
    if language == "Python":
        regex = "def .*\((.*)\).*:"
    elif language == "JavaScript":
        regex = "exports\..* = .*\((.*)\)"
        
    try:
        with open(file) as handle:
            content = handle.read()
    except UnicodeDecodeError as ex:
        return []
        
    matches = re.findall(regex, content)
    split_matches = [match.split(",") for match in matches]
    
    return split_matches
    
def get_file_size(file):
    try:
        with open(file) as f:
            lines = sum(1 for line in f)
    except UnicodeDecodeError as ex:
        return 0
    
    return lines

def get_comment_amount(file, language):
    try:
        with open(file) as handle:
            content = handle.read()
    except UnicodeDecodeError as ex:
        return 0
    
    if language == "Python":
        comments = len(re.findall("^#.*$", content, flags=re.M))
    elif language == "JavaScript":
        comments = len(re.findall("^\/\*.*$", content, flags=re.M))
        comments += len(re.findall("^\/\/.*$", content, flags=re.M))
    
    return comments

def try_detect_provider(file, language):
    try:
        with open(file) as handle:
            content = handle.read()
    except UnicodeDecodeError as ex:
        return []
    
    if language == "Python":
        if "boto3" in content:
            return "AWS"
        if "azure.functions" in content or "function.json" in file:
            return "AZURE"
        if "google.cloud" in content:
            return "GCP"
    elif language == "JavaScript":
        if "aws-sdk" in content or "exports.handler" in content:
            return "AWS"
        if "azure/functions" in content or "function.json" in file:
            return "AZURE"
        if "google-cloud" in content:
            return "GCP"       
    
    return None
        

def get_metrics(target_repo):
    metricas = {}
    max_depth = 0
    total_folders = 0
    total_files = 0
    code_files_per_folder = []
    name_size_per_func = []
    params_per_func = []
    total_config_files = 0
    amount_external_libs = 0
    amount_lines_per_code_file = []
    amount_lines_per_config_file = []
    amount_funcs_per_file = []
    provider = None
    amount_comments_per_file = []

    for dir_name, subdirs, files in os.walk("repo_teste"):
        ## da pra pensar em excluir o readme e gitignore dos arquivos se relevante
        if ".git" in dir_name:
            continue

        if "dir_name" == "repo_teste":
            metricas["root_folder_amount"] = len(subdirs)
            metricas["root_file_amount"] = len(files)

        depth = len(dir_name.split("/"))-1
        if len(dir_name.split("/"))-1 >= max_depth:
            max_depth = depth

        total_files += len(files)
        total_folders += len(subdirs) if ".git" not in subdirs else len(subdirs)-1

        config_files = [file for file in files if is_config_file(file)]
        total_config_files =+ len(config_files)

        for config_file in config_files:
            amount_lines_per_config_file.append(get_file_size(dir_name + "/" +config_file))
            if provider == None:
                provider = try_detect_provider(dir_name + "/" +config_file, target_repo["language"])

        if target_repo["language"] == "Python":
            code_files = [file for file in files if ".py" in file]

            for code_file in code_files:
                func_names = get_func_names(dir_name + "/" + code_file, "Python")
                func_params = get_func_params(dir_name + "/" + code_file, "Python")
                name_size_per_func = name_size_per_func + [len(name) for name in func_names]
                params_per_func = params_per_func + [len(params) for params in func_params]
                amount_funcs_per_file.append(len(func_names))
                amount_lines_per_code_file.append(get_file_size(dir_name + "/" +code_file))
                amount_comments_per_file.append(get_comment_amount(dir_name + "/" + code_file, "Python"))
                
                if provider == None:
                    provider = try_detect_provider(dir_name + "/" + code_file, "Python")

            amount_code_files = len(code_files)
            code_files_per_folder.append(amount_code_files)

            if "requirements.txt" in files:
                amount_external_libs = get_file_size(dir_name + "/" + 'requirements.txt')

        elif target_repo["language"] == "JavaScript":
            code_files = [file for file in files if ".js" in file ]
            
            for code_file in code_files:
                func_names = get_func_names(dir_name + "/" + code_file, "JavaScript")
                func_params = get_func_params(dir_name + "/" + code_file, "JavaScript")
                name_size_per_func = name_size_per_func + [len(name) for name in func_names]
                params_per_func = params_per_func + [len(params) for params in func_params]
                amount_funcs_per_file.append(len(func_names))
                amount_lines_per_code_file.append(get_file_size(dir_name + "/" +code_file))
                amount_comments_per_file.append(get_comment_amount(dir_name + "/" + code_file, "JavaScript"))
                
                if provider == None:
                    provider = try_detect_provider(dir_name + "/" + code_file, "Python")
            
            amount_code_files = len(code_files)
            code_files_per_folder.append(amount_code_files)
            
            if "package.json" in files:
                amount_external_libs = get_file_size(dir_name + "/" + 'package.json')
            
    metricas["max_depth"] = max_depth
    metricas["total_files"] = total_files
    metricas["total_folders"] = total_folders
    metricas["total_code_files"] = sum(code_files_per_folder)
    metricas["avg_code_files_per_folder"] = metricas["total_code_files"] / metricas["total_folders"] if metricas["total_folders"] > 0 else 0
    metricas["total_funcs"] = len(name_size_per_func)
    metricas["avg_func_name_length"] = sum(name_size_per_func) / len(name_size_per_func) if name_size_per_func else 0
    metricas["avg_func_param_amount"] = sum(params_per_func) / len(params_per_func) if params_per_func else 0
    metricas["external_lib_amount"] = amount_external_libs
    metricas["avg_lines_code_files"] = sum(amount_lines_per_code_file) / len(amount_lines_per_code_file) if amount_lines_per_code_file else 0
    metricas["total_code_lines"] = sum(amount_lines_per_code_file)
    metricas["avg_lines_config_files"] = sum(amount_lines_per_config_file) / len(amount_lines_per_config_file) if amount_lines_per_config_file else 0
    
    # Validações estendidas para max e min em listas vazias
    metricas["max_code_files_per_folder"] = max(code_files_per_folder) if code_files_per_folder else 0
    metricas["min_code_files_per_folder"] = min(code_files_per_folder) if code_files_per_folder else 0
    metricas["max_func_name_length"] = max(name_size_per_func) if name_size_per_func else 0
    metricas["min_func_name_length"] = min(name_size_per_func) if name_size_per_func else 0
    metricas["max_lines_code_files"] = max(amount_lines_per_code_file) if amount_lines_per_code_file else 0
    metricas["min_lines_code_files"] = min(amount_lines_per_code_file) if amount_lines_per_code_file else 0
    metricas["max_func_param_amount"] = max(params_per_func) if params_per_func else 0
    metricas["min_func_param_amount"] = min(params_per_func) if params_per_func else 0
    metricas["max_lines_config_files"] = max(amount_lines_per_config_file) if amount_lines_per_config_file else 0
    metricas["min_lines_config_files"] = min(amount_lines_per_config_file) if amount_lines_per_config_file else 0
    
    metricas["avg_funcs_per_code_file"] = metricas["total_funcs"] / metricas["total_code_files"] if metricas["total_code_files"] > 0 else 0
    metricas["max_funcs_per_code_file"] = max(amount_funcs_per_file) if amount_funcs_per_file else 0
    metricas["min_funcs_per_code_file"] = min(amount_funcs_per_file) if amount_funcs_per_file else 0
    
    metricas["avg_comments_per_file"] = sum(amount_comments_per_file) / len(amount_comments_per_file) if amount_comments_per_file else 0
    metricas["max_comments_per_file"] = max(amount_comments_per_file) if amount_comments_per_file else 0
    metricas["min_comments_per_file"] = min(amount_comments_per_file) if amount_comments_per_file else 0
    
    metricas["total_comments"] = sum(amount_comments_per_file)
    metricas["total_config_files"] = len(amount_lines_per_config_file)
    metricas["provider"] = provider
    metricas["language"] = target_repo["language"]
    
    return metricas




    return metricas
        
def clone_repo(repo_url):
    repo = Repo.clone_from(target_repo["clone_url"], "./repo_teste")
    repo
    
def onerror(func, path, exc_info):
    """
    Error handler for ``shutil.rmtree``.

    If the error is due to an access error (read only file)
    it attempts to add write permission and then retries.

    If the error is for another reason it re-raises the error.
    
    Usage : ``shutil.rmtree(path, onerror=onerror)``
    """
    import stat
    # Is the error an access error?
    if not os.access(path, os.W_OK):
        os.chmod(path, stat.S_IWUSR)
        func(path)
    else:
        raise
    
def clear_repo():
    shutil.rmtree("./repo_teste", onerror=onerror)

## Execução de extração para repositórios selecionados

In [ ]:
import pandas as pd
df_metricas = pd.DataFrame()

In [ ]:
target_repos = repos_info

In [ ]:
import os

# Limpa resquícios da pasta temporária se houver falhas anteriores
if os.path.exists("./repo_teste"):
    clear_repo()

i = 0
lista_entradas = []

for target_repo in target_repos:
    i += 1
    print(f"({i}/{len(target_repos)}) clonando {target_repo['clone_url']}")
    clone_repo([target_repo["clone_url"]])
    metricas = get_metrics(target_repo)
    entrada = {**target_repo, **metricas}
    lista_entradas.append(entrada)
    
    # Limpa o repositório clonado para o próximo da lista
    clear_repo()
    
df_metricas = pd.DataFrame(lista_entradas)


In [ ]:
df_metricas = df_metricas.round(decimals=1)
# Exportacao em formato excel caso necessario
# df_metricas.to_csv("metricas_repos.csv", sep=";", index=False)
# print(df_metricas.to_csv(sep=";", index=False))
df_metricas